In [2]:
import numpy as np
from bokeh.io import show, output_notebook
from bokeh.plotting import figure, gridplot

output_notebook()

Loading BokehJS ...

In [3]:
BATCH_SIZE = 20
NB_SAMPLES = 10

In [4]:
mean = np.array([3., 3.])
cov = np.array([[2., 1.5], [1.5, 2.]])
data = np.random.multivariate_normal(mean, cov, size=100)

fig = figure(
    title='Linear subspace',
    width=400,
    height=400,
    x_range=(-5, 10),
    y_range=(-5, 10),
)
fig.circle(data[:, 0], data[:, 1])
show(fig)

data_lin = data
data_lin_backup = np.copy(data_lin)


In [5]:
mean = np.array([3., 2.])
cov = np.array([[2., 0.], [0., .2]])
#x = np.linspace(-5, 5, 1000)
data = np.random.multivariate_normal(mean, cov, size=100)
data[:, 1] = data[:, 1] + (data[:, 0]-mean[0])**2

fig = figure(
    title='Non-Linear subspace',
    width=400,
    height=400,
    x_range=(-5, 10),
    y_range=(-5, 10),
)
fig.circle(data[:, 0], data[:, 1])
x = np.linspace(-1, 7, 1000)
fig.line(x, (x-3)**2+2, color='red')
show(fig)

data_nonlin = data

In [6]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import math

def plot_loss(model, fig,  orig_data, nb_samples=5, box=((-5, -5),(10.1, 10.1)), resolution=100):
    x_min = box[0][0]
    x_max = box[1][0]
    x_range = (x_max - x_min)
    x_res = x_range/resolution
    y_min = box[0][1]
    y_max = box[1][1]
    y_range = (y_max - y_min)
    y_res = y_range/resolution
    xy = np.mgrid[x_min:x_max:x_res, y_min:y_max:y_res].reshape(2,-1).T
    
    losses = []
    model.eval()
    for X in xy:
        X = Variable(torch.from_numpy(X).contiguous().float())
        X = torch.stack([X]*nb_samples)
        obs_mu, obs_logvar,  mu, logvar = model(X)
        loss, _, _ = loss_function(X, obs_mu, obs_logvar, mu, logvar)
        losses.append(math.exp(-loss.data.numpy()[0]))
    
    print(len(losses), type(losses[0]))
       
    losses_norm = mpl.colors.Normalize()(losses)
    colors = [
        "#%02x%02x%02x" % (int(r), int(g), int(b)) 
        for r, g, b, _ in 255*plt.cm.Greys(losses_norm)
    ]
    
    fig.circle(xy[:, 0], xy[:, 1], fill_color=colors, line_color=colors, radius=min(x_range, y_range)/resolution/1.7)
    fig.cross(orig_data[:, 0], orig_data[:, 1], line_color='red')

In [7]:
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.autograd import Variable

class VAE(nn.Module):
    def __init__(self, latent_dim, nb_hidden):
        super(VAE, self).__init__()

        self.enc_in2hid = nn.Linear(2, nb_hidden)
        self.enc_hid2hid = nn.Linear(nb_hidden, nb_hidden)
        self.enc_hid2mu = nn.Linear(nb_hidden, latent_dim)
        self.enc_hid2var = nn.Linear(nb_hidden, latent_dim)
        
        self.dec_lat2hid = nn.Linear(latent_dim, nb_hidden)
        self.dec_hid2hid = nn.Linear(nb_hidden, nb_hidden)
        self.dec_hid2mu = nn.Linear(nb_hidden, 2)
        self.dec_hid2prec = nn.Linear(nb_hidden, 2)

        self.nonlin = F.relu

    def encode(self, x):
        h1 = self.nonlin(self.enc_in2hid(x))
        h2 = self.nonlin(self.enc_hid2hid(h1))
        return self.enc_hid2mu(h2), self.enc_hid2var(h2)

    def reparameterize(self, mu, logvar):
        if self.training:
          std = logvar.mul(0.5).exp_()
          eps = Variable(std.data.new(std.size()).normal_())
          return eps.mul(std).add_(mu)
        else:
          return mu

    def decode(self, z):
        h1 = self.nonlin(self.dec_lat2hid(z))
        h2 = self.nonlin(self.dec_hid2hid(h1))
        return self.dec_hid2mu(h2), self.dec_hid2prec(h2)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        obs_mu, obs_logprec = self.decode(z)
        return obs_mu, obs_logprec, mu, logvar

In [8]:
def gauss_LLH(x, obs_mu, obs_logprec):
    #LLH = Variable(torch.FloatTensor([0.0]))
    #for o_mu, o_logprec, x in zip(obs_mu, obs_logprec, x):
    #    LLH += 0.5*o_logprec.sum() - 0.5*((x-o_mu) * o_logprec.exp() * (x-o_mu)).sum()       
    LLH = (0.5*obs_logprec - 0.5*((x-obs_mu) * obs_logprec.exp() * (x-obs_mu))).sum()
    return LLH / NB_SAMPLES

def isotropic_LLH(x, obs_mu, obs_logvar):
    LLH = Variable(torch.FloatTensor([0.0]))
    for o_mu, o_logvar, x in zip(obs_mu, obs_logvar, x):
        LLH += - 0.5*((x-o_mu) * (x-o_mu)).sum()
    return LLH / NB_SAMPLES

def st_norm_KLD(z_mu, z_logvar, log_sigma=0.0):
    log_sigma = Variable(torch.FloatTensor([log_sigma]))
    KLD = 0.5 * torch.sum(log_sigma - 1 - z_logvar + z_logvar.exp()/log_sigma.exp() + z_mu.pow(2)/log_sigma.exp() )
    return KLD / NB_SAMPLES
    
def loss_function(x, obs_mu, obs_logprec, z_mu, z_logvar, log_sigma=0.0):
    LLH = gauss_LLH(x, obs_mu, obs_logprec)  
    KLD = st_norm_KLD(z_mu, z_logvar, log_sigma)

    return -(LLH - KLD), LLH, KLD

In [9]:
def run_training(training_data, model, optimizer, nb_epochs, elbos, klds, log_interval=10):
    model.train()
    for epoch_no in range(1, nb_epochs+1):
        epoch_data = np.copy(training_data)
        np.random.shuffle(epoch_data)
        epoch_data_batched = epoch_data.reshape(-1, BATCH_SIZE, 2)
        epoch_elbo = 0
        epoch_kld = 0
        epoch_llh = 0
        for X in epoch_data_batched:
            X = Variable(torch.from_numpy(X).float())
            X = torch.cat([X] * NB_SAMPLES)
            optimizer.zero_grad()
            obs_mu, obs_logprec, z_mu, z_logvar = model(X)
            loss, llh, kld = loss_function(X, obs_mu, obs_logprec, z_mu, z_logvar, log_sigma=0.0)
            loss.backward()
            optimizer.step()

            epoch_elbo += -loss.data[0]
            epoch_kld += kld.data[0]
            epoch_llh += llh.data[0]
            
        elbos.append(epoch_elbo)
        klds.append(epoch_kld)    

        if epoch_no % log_interval == 0:
            print('Train Epoch: {} \telbo: {:.6f} \tllh: {:.6f}\tkld: {:.6f}'.format(
                epoch_no, epoch_elbo, epoch_llh, epoch_kld)
            )

In [13]:
model_lin = VAE(latent_dim=2, nb_hidden=2)
optim_lin = optim.Adam(model_lin.parameters(), lr=1e-3, weight_decay=1e-6)
elbos_lin = []
klds_lin = []

In [14]:
run_training(data_lin, model_lin, optim_lin, 10000, elbos_lin, klds_lin, 500)

Train Epoch: 500 	elbo: -159.919125 	llh: -159.880083	kld: 0.039043
Train Epoch: 1000 	elbo: -159.954630 	llh: -159.948843	kld: 0.005786
Train Epoch: 1500 	elbo: -159.772068 	llh: -159.771021	kld: 0.001050
Train Epoch: 2000 	elbo: -159.952820 	llh: -159.952793	kld: 0.000026
Train Epoch: 2500 	elbo: -159.887421 	llh: -159.887358	kld: 0.000060
Train Epoch: 3000 	elbo: -159.965611 	llh: -159.965590	kld: 0.000020
Train Epoch: 3500 	elbo: -159.891239 	llh: -159.891235	kld: 0.000006
Train Epoch: 4000 	elbo: -159.868286 	llh: -159.868235	kld: 0.000050
Train Epoch: 4500 	elbo: -159.983891 	llh: -159.983877	kld: 0.000014
Train Epoch: 5000 	elbo: -159.849068 	llh: -159.849028	kld: 0.000044
Train Epoch: 5500 	elbo: -159.929680 	llh: -159.929676	kld: 0.000005
Train Epoch: 6000 	elbo: -159.898050 	llh: -159.898050	kld: 0.000002
Train Epoch: 6500 	elbo: -132.380896 	llh: -59.154742	kld: 73.226153
Train Epoch: 7000 	elbo: -124.704029 	llh: -34.305202	kld: 90.398826
Train Epoch: 7500 	elbo: -123.24770

In [15]:
lin_box = ((-2,-3), (8,8))
fig1 = figure(
    title='p(x)',
    width=400,
    height=400,
    x_range=(lin_box[0][0], lin_box[1][0]),
    y_range=(lin_box[0][1], lin_box[1][1]),
)
plot_loss(model_lin, fig1, data_lin, nb_samples=NB_SAMPLES, box=lin_box, resolution=150)

fig2 = figure(
    title='ELBO Progress',
    width=400,
    height=400,
)
fig2.line(range(len(elbos_lin)), elbos_lin, color='blue')

fig3 = figure(
    title='KLD Progress',
    width=400,
    height=400,
)
fig3.line(range(len(klds_lin)), klds_lin, color='red')

gp = gridplot([[fig1], [fig2, fig3]])
show(gp)

22500 <class 'float'>


In [17]:
model_nonlin = VAE(latent_dim=2, nb_hidden=40)
optim_nonlin = optim.Adam(model_nonlin.parameters(), lr=1e-3, weight_decay=1e-6)
elbos_nonlin = []
klds_nonlin = []

In [18]:
run_training(data_nonlin, model_nonlin, optim_nonlin, 10000, elbos_nonlin, klds_nonlin, 500)

Train Epoch: 500 	elbo: -93.383078 	llh: 159.906927	kld: 253.290005
Train Epoch: 1000 	elbo: -69.986423 	llh: 224.279678	kld: 294.266102
Train Epoch: 1500 	elbo: -76.260056 	llh: 248.846390	kld: 325.106445
Train Epoch: 2000 	elbo: -49.139263 	llh: 289.877068	kld: 339.016331
Train Epoch: 2500 	elbo: -60.841236 	llh: 286.829723	kld: 347.670959
Train Epoch: 3000 	elbo: -53.132278 	llh: 295.005257	kld: 348.137535
Train Epoch: 3500 	elbo: -49.672714 	llh: 311.266289	kld: 360.939003
Train Epoch: 4000 	elbo: -42.144371 	llh: 322.915466	kld: 365.059837
Train Epoch: 4500 	elbo: -39.439808 	llh: 350.119198	kld: 389.559006
Train Epoch: 5000 	elbo: -35.452003 	llh: 361.173775	kld: 396.625778
Train Epoch: 5500 	elbo: -43.849422 	llh: 345.842060	kld: 389.691483
Train Epoch: 6000 	elbo: -37.653584 	llh: 351.408978	kld: 389.062561
Train Epoch: 6500 	elbo: -46.481102 	llh: 363.274673	kld: 409.755775
Train Epoch: 7000 	elbo: -30.075039 	llh: 368.508579	kld: 398.583618
Train Epoch: 7500 	elbo: -33.835464

In [21]:
nonlin_box =((-0.1, 0.8), (6.1,10.1))
fig1 = figure(
    title='p(x)',
    width=400,
    height=400,
    x_range=(nonlin_box[0][0], nonlin_box[1][0]),
    y_range=(nonlin_box[0][1], nonlin_box[1][1]),
)
plot_loss(model_nonlin, fig1, data_nonlin, nb_samples=NB_SAMPLES, box=nonlin_box, resolution=200)

SUBSAMPLING=1

fig2 = figure(
    title='ELBO Progress',
    width=400,
    height=400,
)
fig2.line(range(len(elbos_nonlin)//SUBSAMPLING), elbos_nonlin[::SUBSAMPLING], color='blue')

fig3 = figure(
    title='KLD Progress',
    width=400,
    height=400,
)
fig3.line(range(len(klds_nonlin)//SUBSAMPLING), klds_nonlin[::SUBSAMPLING], color='red')

gp = gridplot([[fig1], [fig2, fig3]])
show(gp)

40000 <class 'float'>
